# 005-village-crossdate — playground

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import esri_dates, load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.benchmark import error_stats, run_benchmark, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.replay import replay
from uavloc.plotting.correlation import plot_class_match, plot_review
from uavloc.track import experiment_dir, log_run, set_seed

# Params

In [ ]:
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"               # "oldest" (as 005.1-005.4) | "newest" | an exact date, e.g. "2025-04-24"
METHODS = ["segmentation", "vgg11"]
MATCHING = {"segmentation": "weighted-ncc", "vgg11": "cosine"}
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
SEARCH_SCALE = 1.7                # satellite patch = footprint x this
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_DIRECTIONS = 8
SEED = 42
MIN_PEAK_Z = None                 # keep the estimate when the peak z is below this; None = always move

SHOW_FRAME = "birky-15"           # the frame drawn step by step, at PRIMARY_NOISE_M
SHOW_DIRECTION = 2                # which of the N_DIRECTIONS draws
SHOW_METHOD = "segmentation"
OVERVIEW_MARGIN_M = 50.0

LOG_RUN = False                   # True appends each method as a run of 005 (runs.jsonl)
NOTE = ""
DEVICE = "mps"

FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"
EXP = "005-village-crossdate"

# Data

In [ ]:
set_seed(SEED)

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
{place: {"frames": len(places[place]), "dates": esri_dates(place)} for place in PLACES}

# Model

In [ ]:
def run_config(method: str) -> dict:
    return {
        "method": method, "matching": MATCHING[method], "class_weights": CLASS_WEIGHTS,
        "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
        "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
        "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
        "vgg11_checkpoint": VGG11_CHECKPOINT, "drone_input_px": DRONE_INPUT_PX,
        "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
        "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
        "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
        "places": PLACES, "map_date": MAP_DATE, "min_peak_z": MIN_PEAK_Z,
    }

In [ ]:
correlators = {method: {place: PositionCorrelator(build_features(run_config(method), maps[place], DEVICE),
                                                  correlator_config(run_config(method)))
                        for place in PLACES}
               for method in METHODS}

# Run

In [ ]:
def run_method(method: str) -> pd.DataFrame:
    return pd.concat([run_benchmark(correlators[method][place], places[place], noise_m, N_DIRECTIONS, SEED).assign(place=place)
                      for place in PLACES for noise_m in NOISE_M], ignore_index=True)

In [ ]:
def gated(records: pd.DataFrame, min_peak_z: float | None) -> pd.DataFrame:
    """Draws whose peak z is below `min_peak_z` keep the estimate: their error becomes the injected one."""
    if min_peak_z is None:
        return records
    keep = records["peak_z"].fillna(0) < min_peak_z
    return records.assign(error_m=np.where(keep, records["prior_error_m"], records["error_m"]), kept_estimate=keep)

In [ ]:
records = {method: gated(run_method(method), MIN_PEAK_Z) for method in METHODS}

# Results

In [ ]:
pd.DataFrame({(method, noise_m): error_stats(r[r["noise_m"] == noise_m])
              for method, r in records.items() for noise_m in NOISE_M}).T[
    ["mean_position_error_m", "median_position_error_m", "p90_position_error_m", "improved_share"]].round(2)

In [ ]:
pd.DataFrame({method: r[r["noise_m"] == PRIMARY_NOISE_M].groupby("place")["error_m"].mean()
              for method, r in records.items()}).round(2)

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(7, 4))
for method, r in records.items():
    errors = np.sort(r[r["noise_m"] == PRIMARY_NOISE_M]["error_m"].to_numpy())
    ax.plot(errors, np.linspace(0, 1, len(errors)), label=method)
ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1, label="no correction")
ax.set_xlabel(f"error after correction @ {PRIMARY_NOISE_M:g} m, m"); ax.set_ylabel("share of draws")
ax.set_title(f"map {MAP_DATE}, scale {SEARCH_SCALE}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
per_frame = pd.DataFrame({method: r[r["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean()
                          for method, r in records.items()}).sort_values(METHODS[0])
fig_frames, ax = plt.subplots(figsize=(14, 4))
per_frame.plot.bar(ax=ax, width=.8)
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title("per frame, 8 directions each")
fig_frames.tight_layout()

In [ ]:
frame = next(f for f in frames if f.name == SHOW_FRAME)
draw = records[SHOW_METHOD].query("frame == @SHOW_FRAME and noise_m == @PRIMARY_NOISE_M").iloc[SHOW_DIRECTION]
shown = replay(correlators[SHOW_METHOD][frame_place(frame)], frame, LatLon(draw["prior_lat"], draw["prior_lon"]))
fig_case = plot_review(shown, maps[frame_place(frame)],
                       f"{SHOW_METHOD}: {SHOW_FRAME} | estimate {draw['prior_error_m']:.1f} m off -> {draw['error_m']:.2f} m",
                       f"map {MAP_DATE}", OVERVIEW_MARGIN_M)

In [ ]:
if SHOW_METHOD == "segmentation":
    fig_match = plot_class_match(shown, correlators[SHOW_METHOD][frame_place(frame)].features, CLASS_WEIGHTS)

# Log

In [ ]:
if LOG_RUN:
    for method, r in records.items():
        run = log_run(exp=EXP, config=run_config(method), metrics=summarise(r, {"all": {f.name for f in frames}}, PRIMARY_NOISE_M),
                      note=NOTE or f"playground: {method}, map {MAP_DATE}, scale {SEARCH_SCALE}", figures=[fig_cdf, fig_frames],
                      seed=SEED, device=DEVICE)
        r.to_csv(experiment_dir(EXP) / "artifacts" / f"records-{run['run']}.csv", index=False)